# California / Hawaii NFDRS Normals 2016–2025 — by RAWS, PSA and FDRA

Builds the "what's normal" tables for the CA Fire Weather Experience Builder app.

- **Stations:** the RAWS in the live *CA NFDRS 3.0 (FEMS)* layer that the app displays (California + Hawaii),
  so normals line up 1:1 with the live values: same stations, same PSA assignments, same fuel model (Y).
- **Data:** FEMS daily NFDR summaries, **observed** values only, 2016-01-01 → 2025-12-31.
- **PSA / FDRA values:** the daily mean of reporting member stations, the same rule the dashboard uses.
  FDRA membership comes from point-in-polygon against your FDRA layer (stations outside every FDRA,
  e.g. Hawaii if the layer has no HI FDRAs, still get station and PSA normals). An FDRA with no RAWS inside
  uses its nearest station and is flagged `nearest_only = 1`.
- **Indices:** ERC, IC, BI, SC, FM-1, FM-10, FM-100, FM-1000, KBDI, GSI, Woody FM, Herb FM.

**Outputs** (in `normals/output/`), ready to add to ArcGIS Online as hosted tables:

| File | One row per | Use in the app |
|---|---|---|
| `ca_normals_annual_2016_2025.csv` | station / PSA / FDRA | percentile thresholds (p3 … p97) and mean for every index |
| `ca_normals_doy_2016_2025.csv` | PSA / FDRA × day of year | normal band (p10–p90, median, mean) on the season charts |
| `ca_normals_stations_2016_2025.csv` | station | station → PSA → FDRA lookup |
| `ca_fdra_stations_2016_2025.csv` | FDRA | which RAWS feed each FDRA |

Run top to bottom. The download (≈ 2,000 station-years) is cached in `normals/fems_raw/`; if it stops,
just re-run and it picks up where it left off.


In [ ]:
# 1 — Install packages (once)
import subprocess, sys
for pkg in ['pandas', 'numpy', 'requests', 'shapely', 'pyshp', 'pyproj', 'matplotlib', 'python-dotenv', 'arcgis']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], capture_output=True, text=True)
    print(('OK  ' if r.returncode == 0 else 'FAIL') + f' {pkg}')


In [ ]:
# 2 — Settings  (edit these)
from pathlib import Path
START_YEAR = 2016
END_YEAR   = 2025

HERE     = Path.cwd()                       # run from the repo's normals/ folder
RAW_DIR  = HERE / 'fems_raw'                # cached FEMS downloads (git-ignored)
OUT_DIR  = HERE / 'output'

# FDRA polygons — option A (default): read the FDRA_CA item in NIFC AGOL directly.
#   Signs in through ArcGIS Pro on this PC, so Pro must be installed and signed in to NIFC.
FDRA_ITEM_ID     = 'd18bbbbcb5704673970fb7889da997a6'   # FDRA_CA (CAL FIRE FDRA_View)
# Option B: a shapefile (.zip) or GeoJSON exported from that layer; used if option A fails.
FDRA_FILE        = HERE / 'fdra' / 'FDRA_CA.zip'
FDRA_NAME_FIELDS = ['UNIT', 'FDRA']         # FDRA id = UNIT-FDRA, e.g. BEU-2

INCLUDE_STATION_DOY = False                 # day-of-year normals per station (large file)
WORKERS = 8                                 # parallel FEMS requests


In [ ]:
# 3 — FEMS credentials
# Put these two lines in a file named .env in the repo root (it is git-ignored):
#   FEMS_API_KEY=your_key
#   FEMS_USERNAME=your.email@usda.gov
import os, getpass
from dotenv import load_dotenv
load_dotenv(HERE.parent / '.env'); load_dotenv(HERE / '.env')
FEMS_API_KEY  = os.getenv('FEMS_API_KEY')  or getpass.getpass('FEMS API key: ')
FEMS_USERNAME = os.getenv('FEMS_USERNAME') or input('FEMS username (email): ')
print('FEMS credentials loaded for', FEMS_USERNAME)


In [ ]:
# 4 — Stations and PSAs from the live CA NFDRS 3.0 layer
import logging, importlib
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(message)s', datefmt='%H:%M:%S', force=True)
import ca_normals as cn; importlib.reload(cn)

stations = cn.load_ca_stations()
psas     = cn.load_ca_psas()
print(stations.groupby('GACC')['PSA'].nunique().rename('PSAs').to_frame().join(
      stations.groupby('GACC').size().rename('RAWS')))
stations.head()


In [ ]:
# 5 — Assign stations to FDRAs
fdra_tbl, fdras = None, None
if FDRA_ITEM_ID:
    try:
        fdras = cn.load_fdra_from_agol(FDRA_ITEM_ID, FDRA_NAME_FIELDS)
    except Exception as e:
        print('Could not read FDRA_CA from AGOL via ArcGIS Pro:', e)
if fdras is None and FDRA_FILE and Path(FDRA_FILE).exists():
    fdras = cn.load_fdra_polygons(FDRA_FILE, FDRA_NAME_FIELDS)
if fdras:
    stations, fdra_tbl = cn.assign_fdras(stations, fdras)
    display(fdra_tbl[['fdra_id', 'n_stations', 'nearest_only']].sort_values('fdra_id').head(20))
    print('Stations outside every FDRA (Hawaii, etc.):', stations['FDRA'].isna().sum())
else:
    print('No FDRA polygons — skipping FDRA normals. Set FDRA_ITEM_ID or FDRA_FILE in cell 2.')


In [ ]:
# 6 — Download FEMS history (resumable; ~20–40 min the first time)
failed = cn.download_history(stations, START_YEAR, END_YEAR, RAW_DIR, FEMS_API_KEY, FEMS_USERNAME, WORKERS)
print('Failed station-years:', len(failed), '(re-run this cell to retry)' if failed else '')


In [ ]:
# 7 — Load observed history and check coverage
hist = cn.load_history(stations, START_YEAR, END_YEAR, RAW_DIR)
cov = (hist.assign(year=hist['date'].dt.year)
           .groupby('year').agg(stations=('stationId', 'nunique'), station_days=('erc', 'count')))
display(cov)
low = (hist.groupby('stationId')['erc'].count() / (365.25 * (END_YEAR - START_YEAR + 1)))
print(f'Stations with < 50% of days: {(low < 0.5).sum()}  →', low[low < 0.5].sort_values().head(15).round(2).to_dict())


In [ ]:
# 8 — Aggregate and compute normals
import pandas as pd
membership = {
    'STATION': {str(s): [s] for s in stations['stationId']},
    'PSA':     stations.groupby('PSA')['stationId'].apply(list).to_dict(),
}
if fdra_tbl is not None:
    membership['FDRA'] = dict(zip(fdra_tbl['fdra_id'], fdra_tbl['stations']))

daily = pd.concat([cn.unit_daily(hist, m, ut) for ut, m in membership.items()], ignore_index=True)
annual = cn.annual_normals(daily)

doy_src = daily if INCLUDE_STATION_DOY else daily[daily['unit_type'] != 'STATION']
doy = cn.doy_normals(doy_src, plot_year=END_YEAR + 1)
print(annual.groupby('unit_type').size().to_dict(), '| DOY rows:', len(doy))
annual[annual['unit_type'] == 'PSA'][['unit_id', 'n_days', 'erc_mean', 'erc_p90', 'erc_p97',
                                     'fm100_p3', 'fm100_p10', 'ic_p90', 'sc_p90']].head(12)


In [ ]:
# 9 — Write outputs
files = cn.write_outputs(OUT_DIR, START_YEAR, END_YEAR, stations, fdra_tbl, annual, doy)
for k, f in files.items():
    print(f'{k:9s} {f}')


In [ ]:
# 10 — Quick look: ERC and FM-100 normals for one PSA
import matplotlib.pyplot as plt
PSA = 'SC10'
d = doy[(doy.unit_type == 'PSA') & (doy.unit_id == PSA)]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, k in zip(axes, ['erc', 'fm100']):
    ax.fill_between(d.plot_date, d[f'{k}_p10'], d[f'{k}_p90'], alpha=.25, label='p10–p90')
    ax.plot(d.plot_date, d[f'{k}_p50'], lw=1.5, label='median')
    ax.set_title(f'{PSA} {k.upper()} normals {START_YEAR}–{END_YEAR}'); ax.legend()
plt.tight_layout(); plt.show()


## Add the tables to ArcGIS Online (NIFC)

For `ca_normals_annual_2016_2025.csv` and `ca_normals_doy_2016_2025.csv` (and the two lookup CSVs):

1. **Content → New item → Your device**, choose the CSV.
2. Choose **Add … and create a hosted table** (no location). Keep the field types it suggests;
   make sure `plot_date` is a **Date** field.
3. Title them `CA NFDRS Normals 2016–2025 – Annual` and `CA NFDRS Normals 2016–2025 – Day of Year`, folder
   *GACC Fire Weather*, tag `ca-fire-wx`.

These are static. Re-run this notebook and overwrite the tables (item → **Update data → Overwrite**) only
when you change the period or the station set.
